## 1. Importar bibliotecas e carregar o dataset


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

print("Bibliotecas importadas com sucesso!")

# Dataset baseado no da Semana 8 (tamanho x preço), agora estendido
# com mais variáveis para a regressão linear MÚLTIPLA
np.random.seed(42)

tamanho_m2 = [50, 60, 70, 80, 90, 100, 110, 120, 130, 140,
              150, 160, 170, 180, 190, 200]

n = len(tamanho_m2)
quartos = [2, 2, 2, 3, 3, 3, 3, 4, 4, 4, 4, 5, 5, 5, 5, 6]
idade_anos = [20, 18, 15, 25, 10, 8, 30, 5, 12, 22, 3, 17, 9, 28, 6, 14]
distancia_centro_km = [12, 10, 9, 15, 7, 6, 18, 4, 8, 16, 3, 11, 5, 20, 4, 9]

# Fórmula combinando as variáveis (base: 3 mil R$/m², como na semana 8)
# + valoriza quartos, + desvaloriza idade e distância do centro
preco_mil = (
    3.0 * np.array(tamanho_m2)
    + 15.0 * np.array(quartos)
    - 1.5 * np.array(idade_anos)
    - 4.0 * np.array(distancia_centro_km)
    + np.random.normal(0, 5, n)  # pequeno ruído para ficar mais realista
).round(1)

df = pd.DataFrame({
    'tamanho_m2': tamanho_m2,
    'quartos': quartos,
    'idade_anos': idade_anos,
    'distancia_centro_km': distancia_centro_km,
    'preco_mil': preco_mil
})

print("\n📊 Dataset de casas (múltiplas variáveis):")
print(df)

print(f"\n🔹 Total de registros: {len(df)}")
print(f"🔹 Variáveis disponíveis: {list(df.columns)}")


Bibliotecas importadas com sucesso!

📊 Dataset de casas (múltiplas variáveis):
    tamanho_m2  quartos  idade_anos  distancia_centro_km  preco_mil
0           50        2          20                   12      104.5
1           60        2          18                   10      142.3
2           70        2          15                    9      184.7
3           80        3          25                   15      195.1
4           90        3          10                    7      270.8
5          100        3           8                    6      307.8
6          110        3          30                   18      265.9
7          120        4           5                    4      400.3
8          130        4          12                    8      397.7
9          140        4          22                   16      385.7
10         150        4           3                    3      491.2
11         160        5          17                   11      483.2
12         170        5           9  

## 2. Separar treino/teste, treinar o modelo e avaliar

Usamos `test_size=0.2` (80% treino / 20% teste), como na semana 8, agora com 4 variáveis independentes (`tamanho_m2`, `quartos`, `idade_anos`, `distancia_centro_km`).

In [3]:
# Separar variáveis independentes (X) e variável dependente (y)
X = df[['tamanho_m2', 'quartos', 'idade_anos', 'distancia_centro_km']]
y = df['preco_mil']

print(f"🔹 X (variáveis independentes): {X.shape[1]} colunas, {X.shape[0]} registros")
print(f"🔹 y (preço): {y.shape[0]} registros")

# Dividir em treino (80%) e teste (20%)
X_treino, X_teste, y_treino, y_teste = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"\n📚 Dados de treino: {len(X_treino)} casas")
print(f"🧪 Dados de teste: {len(X_teste)} casas")

# Criar e treinar o modelo de regressão linear múltipla
modelo = LinearRegression()
modelo.fit(X_treino, y_treino)

print("\n✅ Modelo treinado com sucesso!")

# Fazer previsões com os dados de teste
previsoes = modelo.predict(X_teste)

print("\n📋 Comparação entre valores reais e previstos:")
comparacao = pd.DataFrame({
    'Preço Real (mil R$)': y_teste.values,
    'Preço Previsto (mil R$)': previsoes.round(1)
})
print(comparacao)

# Avaliar o modelo com métricas
mae = mean_absolute_error(y_teste, previsoes)
mse = mean_squared_error(y_teste, previsoes)
r2 = r2_score(y_teste, previsoes)

print("\n📊 Métricas de Avaliação:")
print(f"   - MAE (erro absoluto médio): {mae:.2f}")
print(f"   - MSE (erro quadrático médio): {mse:.2f}")
print(f"   - R² (coeficiente de determinação): {r2:.4f}")

# Mostrar os coeficientes de cada variável
print("\n📐 Coeficientes do modelo (impacto de cada variável no preço):")
for nome_var, coef in zip(X.columns, modelo.coef_):
    print(f"   - {nome_var}: {coef:.2f}")
print(f"   - Intercepto: {modelo.intercept_:.2f}")


🔹 X (variáveis independentes): 4 colunas, 16 registros
🔹 y (preço): 16 registros

📚 Dados de treino: 12 casas
🧪 Dados de teste: 4 casas

✅ Modelo treinado com sucesso!

📋 Comparação entre valores reais e previstos:
   Preço Real (mil R$)  Preço Previsto (mil R$)
0                104.5                    109.8
1                142.3                    151.5
2                307.8                    309.7
3                611.4                    616.8

📊 Métricas de Avaliação:
   - MAE (erro absoluto médio): 5.44
   - MSE (erro quadrático médio): 36.14
   - R² (coeficiente de determinação): 0.9991

📐 Coeficientes do modelo (impacto de cada variável no preço):
   - tamanho_m2: 2.93
   - quartos: 15.52
   - idade_anos: -0.14
   - distancia_centro_km: -6.05
   - Intercepto: 7.70


## 3. Respondendo às perguntas da atividade

**Quais foram os valores de MAE, MSE e R²?**
MAE ≈ 5,44 mil reais, MSE ≈ 36,14 e R² ≈ 0,9991.

**O que o R² indica sobre a qualidade do modelo?**
Um R² de 0,9991 significa que o modelo explica cerca de 99,9% da variação dos preços, ou seja, é um ajuste excelente.

**Quais variáveis tiveram coeficientes positivos? E negativos?**
`tamanho_m2` (+2,93) e `quartos` (+15,52) tiveram coeficientes positivos, enquanto `idade_anos` (−0,14) e `distancia_centro_km` (−6,05) tiveram coeficientes negativos.

**Qual variável parece influenciar mais o preço?**
O número de `quartos` é a variável com maior impacto por unidade (+15,52 mil R$ por quarto adicional), seguida de perto pelo efeito negativo da `distancia_centro_km`.
